# 04 — Geomecanica QA: Modelo de Terra Mecanico (MEM)

Este notebook e um estudo de caso aprofundado no modulo de **Geomecanica** do Geolytics Dictionary. Cobrimos:

- Consulta a propriedades elasticas necessarias para um MEM 1D.
- Visualizacao do **Circulo de Mohr** para um estado de tensao sintetico.
- Validacao SHACL da restricao `StressTensorShape`.
- Mapeamento de fraturas para classes GSO/Loop3D via crosswalk.

## Pre-requisitos

```bash
pip install requests matplotlib
# Opcional para validacao SHACL completa:
# pip install pyshacl rdflib
```

## Objetivo

1. Carregar `geomechanics.json` e `geomechanics-fractures.json`.
2. Consultar quais propriedades elasticas sao necessarias para um MEM 1D.
3. Plotar um Circulo de Mohr com envelope de Mohr-Coulomb.
4. Demonstrar a restricao SHACL `StressTensorShape` (um valido, um invalido).
5. Realizar o crosswalk fratura -> GSO.

## 1. Carregamento dos dados

In [ ]:
import json
import os
import math
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

BASE_URL = 'https://thiagoflc.github.io/geolytics-dictionary'
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

def fetch_json(url: str, local_path: str) -> dict:
    """Baixa JSON via HTTP com fallback para arquivo local."""
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        print(f'Carregado via HTTP: {url.split("/")[-1]}')
        return resp.json()
    except Exception as exc:
        print(f'HTTP falhou ({exc}), usando local: {local_path}')
        with open(local_path, encoding='utf-8') as fh:
            return json.load(fh)

geomec = fetch_json(
    f'{BASE_URL}/data/geomechanics.json',
    os.path.join(DATA_DIR, 'geomechanics.json'),
)
geomec_frac = fetch_json(
    f'{BASE_URL}/data/geomechanics-fractures.json',
    os.path.join(DATA_DIR, 'geomechanics-fractures.json'),
)
frac_gso = fetch_json(
    f'{BASE_URL}/data/fracture_to_gso.json',
    os.path.join(DATA_DIR, 'fracture_to_gso.json'),
)
taxonomies = fetch_json(
    f'{BASE_URL}/data/taxonomies.json',
    os.path.join(DATA_DIR, 'taxonomies.json'),
)

print(f'\ngeomechanics.json   : {geomec["meta"]["class_count"]} classes, {geomec["meta"]["property_count"]} propriedades')
print(f'geomechanics-fractures: {len(geomec_frac["classes"])} classes')
print(f'fracture_to_gso     : {len(frac_gso)} mapeamentos')
# Saida esperada:
# geomechanics.json   : 26 classes, 22 propriedades
# geomechanics-fractures: 17 classes
# fracture_to_gso     : 8 mapeamentos

## 2. Consulta: propriedades elasticas para um MEM 1D

In [ ]:
# Um MEM 1D requer quatro pilares: tensao in situ, pressao de poros,
# modulos elasticos e geometria de contencao.
# Aqui identificamos as propriedades elasticas (Pilar 3) a partir da ontologia.

def query_properties_by_class(data: dict, class_name: str) -> list[dict]:
    """Retorna propriedades da ontologia associadas a uma dada classe."""
    props = data.get('properties', [])
    return [p for p in props if class_name.lower() in p.get('domain', '').lower()]

# Classes de modulos elasticos no geomechanics.json
ELASTIC_CLASSES = ['ElasticModuli', 'YoungModulus', 'PoissonRatio', 'ShearModulus', 'BulkModulus']

print('Propriedades elasticas para MEM 1D:')
print('=' * 60)

all_classes = {c['name']: c for c in geomec.get('classes', [])}
elastic_classes_found = [
    c for c in geomec.get('classes', [])
    if any(ec.lower() in c['name'].lower() for ec in ELASTIC_CLASSES)
]

for cls in elastic_classes_found:
    print(f"\nClasse: {cls['name']} (id={cls['id']})")
    print(f"  PT   : {cls.get('name_pt', 'N/A')}")
    print(f"  Descr: {cls['description'][:100]}...")
    props_for_cls = [p for p in geomec.get('properties', []) if cls['name'] in p.get('domain', '')]
    if props_for_cls:
        print(f"  Propriedades: {', '.join(p['name'] for p in props_for_cls)}")
# Saida esperada:
# Classe: ElasticModuli (id=GM006)
#   PT   : ModulosElasticos
#   Descr: Collection of elastic moduli...
# Classe: YoungModulus (id=GM007)
# Classe: PoissonRatio (id=GM008)
# ...

In [ ]:
# Resumo dos modulos necessarios para MEM 1D
print('Resumo — Pilar 3 (Modulos Elasticos) para MEM 1D:')
print()
mem_pillars = [
    ('Pilar 1', 'Tensao in situ', 'StressTensor, PrincipalStress, HorizStressRatio'),
    ('Pilar 2', 'Pressao de poros', 'PorePressure, EffectiveStress, BiotCoefficient'),
    ('Pilar 3', 'Modulos elasticos', 'YoungModulus [GPa], PoissonRatio [-], ShearModulus [GPa], BulkModulus [GPa]'),
    ('Pilar 4', 'Geometria de contencao', 'ContainmentGeometry, FracturePressure, CollapseGradient'),
]

for pilar, nome, props in mem_pillars:
    print(f'{pilar}: {nome}')
    print(f'  Propriedades: {props}')
    print()
# Saida esperada: tabela dos 4 pilares do MEM com propriedades

## 3. Circulo de Mohr

In [ ]:
# Estado de tensao sintetico (rocha de reservatorio tipica pre-sal)
# Convencao: sigma_1 >= sigma_2 >= sigma_3 (compressao positiva)

sigma_1 = 60.0  # MPa — tensao principal maxima (litostática)
sigma_2 = 45.0  # MPa — tensao principal intermediaria
sigma_3 = 30.0  # MPa — tensao principal minima (horizontal minima)
pore_p  = 25.0  # MPa — pressao de poros

# Tensoes efetivas (principio de Terzaghi)
s1_eff = sigma_1 - pore_p
s3_eff = sigma_3 - pore_p

# Parametros Mohr-Coulomb
cohesion    = 5.0   # MPa
friction_deg = 30.0  # graus
friction_rad = math.radians(friction_deg)

print(f'Tensoes totais : sigma_1={sigma_1}, sigma_2={sigma_2}, sigma_3={sigma_3} MPa')
print(f'Pressao de poros: {pore_p} MPa')
print(f'Tensoes efetivas: sigma_1_eff={s1_eff}, sigma_3_eff={s3_eff} MPa')
print(f'Parametros MC  : c={cohesion} MPa, phi={friction_deg}°')
# Saida esperada:
# Tensoes totais : sigma_1=60.0, sigma_2=45.0, sigma_3=30.0 MPa
# Pressao de poros: 25.0 MPa
# Tensoes efetivas: sigma_1_eff=35.0, sigma_3_eff=5.0 MPa

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Circulo de Mohr (espaco sigma-tau)
center = (s1_eff + s3_eff) / 2
radius = (s1_eff - s3_eff) / 2
theta = np.linspace(0, 2 * np.pi, 360)
sigma_circ = center + radius * np.cos(theta)
tau_circ   = radius * np.sin(theta)

ax.plot(sigma_circ, tau_circ, 'b-', linewidth=2, label='Circulo de Mohr')
ax.plot(center, 0, 'bo', markersize=4)

# Envelope de Mohr-Coulomb: tau = c + sigma * tan(phi)
sigma_env = np.linspace(-5, s1_eff + 5, 300)
tau_env   = cohesion + sigma_env * math.tan(friction_rad)
ax.plot(sigma_env, tau_env, 'r--', linewidth=1.5,
        label=f'Envoltoria M-C (c={cohesion} MPa, phi={friction_deg}°)')

# Pontos notaveis
ax.plot([s3_eff, s1_eff], [0, 0], 'b^', markersize=8)
ax.annotate(f'sigma_3_eff\n{s3_eff} MPa', xy=(s3_eff, 0),
            xytext=(s3_eff - 4, radius * 0.3),
            fontsize=8, arrowprops=dict(arrowstyle='->'))
ax.annotate(f'sigma_1_eff\n{s1_eff} MPa', xy=(s1_eff, 0),
            xytext=(s1_eff + 1, radius * 0.3),
            fontsize=8, arrowprops=dict(arrowstyle='->'))

ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5, linestyle=':')
ax.set_xlabel('Tensao Normal Efetiva (MPa)', fontsize=11)
ax.set_ylabel('Tensao Cisalhante (MPa)', fontsize=11)
ax.set_title('Circulo de Mohr — Estado de Tensao Sintetico\n'
             f'sigma_1={sigma_1}, sigma_2={sigma_2}, sigma_3={sigma_3} MPa; Pp={pore_p} MPa',
             fontsize=11)
ax.legend(fontsize=9)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Verificacao de estabilidade
tau_max = radius
sigma_at_tau_max = center
tau_mc_at_center = cohesion + sigma_at_tau_max * math.tan(friction_rad)
stable = tau_max < tau_mc_at_center
print(f'tau_max = {tau_max:.2f} MPa')
print(f'Envoltoria M-C em sigma={sigma_at_tau_max:.2f}: tau={tau_mc_at_center:.2f} MPa')
print(f'Estado de tensao ESTAVEL: {stable}')
# Saida esperada:
# Circulo de Mohr plotado — o circulo nao toca a envoltoria M-C
# tau_max = 15.0 MPa
# Estado de tensao ESTAVEL: True

O circulo de Mohr representa o estado de tensao efetivo no espaco tensao-normal / tensao-cisalhante. Quando o circulo toca ou cruza a envoltoria de Mohr-Coulomb, ocorre ruptura da rocha.

## 4. Validacao SHACL: StressTensorShape

In [ ]:
# A restricao StressTensorShape exige: sigma_1 >= sigma_2 >= sigma_3.
# Abaixo demonstramos a logica da restricao sem precisar do pyshacl.

STRESS_SHAPE_RULE = 'StressTensorShape: sigma_1 >= sigma_2 >= sigma_3'

def validate_stress_tensor(s1: float, s2: float, s3: float) -> dict:
    """Valida a restricao de ordenacao do tensor de tensao."""
    violations = []
    if not (s1 >= s2):
        violations.append(f'sigma_1 ({s1}) < sigma_2 ({s2}) — viola convencao')
    if not (s2 >= s3):
        violations.append(f'sigma_2 ({s2}) < sigma_3 ({s3}) — viola convencao')
    return {'valid': len(violations) == 0, 'violations': violations}

# Caso 1: valido
v1 = validate_stress_tensor(sigma_1, sigma_2, sigma_3)
print(f'Caso 1 (sigma_1={sigma_1}, sigma_2={sigma_2}, sigma_3={sigma_3}):')
print(f'  Valido: {v1["valid"]}')
print(f'  Violacoes: {v1["violations"] or "nenhuma"}')

# Caso 2: invalido (ordem incorreta)
v2 = validate_stress_tensor(30.0, 45.0, 60.0)  # ordem invertida
print(f'\nCaso 2 (sigma_1=30, sigma_2=45, sigma_3=60 — ordem invertida):')
print(f'  Valido: {v2["valid"]}')
print(f'  Violacoes: {v2["violations"]}')
# Saida esperada:
# Caso 1 (60, 45, 30): Valido: True  Violacoes: nenhuma
# Caso 2 (30, 45, 60): Valido: False  Violacoes: ['sigma_1 (30) < sigma_2 (45)...']

In [ ]:
# Validacao completa via pyshacl (se disponivel)
SHAPES_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'geolytics-shapes.ttl')
VOCAB_PATH  = os.path.join(os.path.dirname(os.getcwd()), 'data', 'geolytics-vocab.ttl')

try:
    import pyshacl
    from rdflib import Graph, Literal, Namespace, RDF
    from rdflib.namespace import XSD

    GEO = Namespace('https://geolytics.petrobras.com.br/dict/')

    def make_stress_graph(s1: float, s2: float, s3: float) -> Graph:
        g = Graph()
        node = GEO.SyntheticTensor1
        g.add((node, RDF.type, GEO.StressTensor))
        g.add((node, GEO.sigma_1, Literal(s1, datatype=XSD.float)))
        g.add((node, GEO.sigma_2, Literal(s2, datatype=XSD.float)))
        g.add((node, GEO.sigma_3, Literal(s3, datatype=XSD.float)))
        return g

    shapes_graph = Graph().parse(SHAPES_PATH, format='turtle')
    vocab_graph  = Graph().parse(VOCAB_PATH,  format='turtle')

    for label, s1, s2, s3 in [('Valido', 60, 45, 30), ('Invalido', 30, 45, 60)]:
        data_g = make_stress_graph(s1, s2, s3)
        conforms, _, report_text = pyshacl.validate(
            data_g,
            shacl_graph=shapes_graph,
            ont_graph=vocab_graph,
            inference='rdfs',
        )
        print(f'[{label}] sigma_1={s1}, sigma_2={s2}, sigma_3={s3}')
        print(f'  Conforms: {conforms}')
        if not conforms:
            for line in report_text.strip().splitlines()[:5]:
                print(f'  {line}')
        print()

except ImportError:
    print('pyshacl nao disponivel — usando validacao Python nativa (celula anterior)')
    print('Para instalar: pip install pyshacl rdflib')
# Saida esperada (com pyshacl):
# [Valido]   Conforms: True
# [Invalido] Conforms: False
#   Violation of sh:constraint ... sigma_1 < sigma_2

## 5. Crosswalk fratura -> GSO/Loop3D

In [ ]:
print('Mapeamentos Geolytics Fratura -> GSO/Loop3D:')
print('=' * 60)

for mapping in frac_gso:
    geo_id  = mapping.get('geo_id', mapping.get('geolytics_id', '?'))
    geo_nm  = mapping.get('geo_label', mapping.get('geolytics_label', '?'))
    gso_uri = mapping.get('gso_uri', '?')
    gso_nm  = mapping.get('gso_label', '?')
    rel     = mapping.get('skos_relation', mapping.get('relation', '?'))
    print(f'{geo_nm} ({geo_id})')
    print(f'  --[{rel}]--> {gso_nm}')
    print(f'  GSO URI: {gso_uri}')
    print()
# Saida esperada:
# FaultZone (GF002)
#   --[skos:exactMatch]--> gso:FaultZone
#   GSO URI: https://w3id.org/gso/1.0/geology/FaultZone
# ...

In [ ]:
# Visualizacao tabular dos mapeamentos
import collections

rel_types = collections.Counter(
    m.get('skos_relation', m.get('relation', 'unknown')) for m in frac_gso
)

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(rel_types.keys(), rel_types.values(), color=['#378ADD', '#4CAF50', '#F4A522'][:len(rel_types)])
ax.set_title('Tipos de relacao SKOS no crosswalk Fratura -> GSO', fontsize=11)
ax.set_xlabel('Tipo de relacao SKOS')
ax.set_ylabel('Numero de mapeamentos')
plt.tight_layout()
plt.show()
# Saida esperada: grafico de barras com exactMatch, closeMatch, broadMatch

## 6. Consulta textual nas classes de geomecanica

In [ ]:
def search_geomec(query: str, top_k: int = 5) -> list[dict]:
    """Busca textual simples nas classes da ontologia geomecanica."""
    query_lower = query.lower()
    all_classes = geomec.get('classes', []) + geomec_frac.get('classes', [])
    results = []
    for cls in all_classes:
        score = 0
        text = (cls.get('description', '') + ' ' + cls.get('description_pt', '')).lower()
        for token in query_lower.split():
            if token in text:
                score += text.count(token)
        if score > 0:
            results.append({'class': cls['name'], 'id': cls['id'], 'score': score,
                            'desc_pt': cls.get('description_pt', '')[:120]})
    return sorted(results, key=lambda x: -x['score'])[:top_k]

QUERY_MEM = 'propriedades elasticas modulo Young medidas testemunho'
hits = search_geomec(QUERY_MEM)

print(f'Query: "{QUERY_MEM}"')
print('=' * 60)
for h in hits:
    print(f"#{h['id']}  {h['class']}  (score={h['score']})")
    print(f"  {h['desc_pt']}...")
    print()
# Saida esperada:
# #GM007 YoungModulus (score alto)
#   Razao entre tensao axial e deformacao axial...
# #GM008 PoissonRatio
# ...

## Resumo

Neste notebook:

- Identificamos os **4 pilares do MEM 1D** e suas propriedades na ontologia Geolytics.
- Plotamos o **Circulo de Mohr** para um estado de tensao sintetico tipico do pre-sal.
- Demonstramos a restricao SHACL `StressTensorShape` em datasets sinteticos.
- Realizamos o crosswalk de 8 classes de fraturas para o vocabulario GSO/Loop3D.

Para aprofundamento, veja `docs/GEOMECHANICS.md` e `docs/SHACL.md` no repositorio.